In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare spectral features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Savitzky–Golay smoothing
# -----------------------------
def apply_savgol(X, X_test):

    print("Applying Savitzky–Golay smoothing...")

    X_sg = savgol_filter(
        X,
        window_length=11,
        polyorder=2,
        deriv=0,
        axis=1
    )

    X_test_sg = savgol_filter(
        X_test,
        window_length=11,
        polyorder=2,
        deriv=0,
        axis=1
    )

    return X_sg, X_test_sg


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    return X_scaled, X_test_scaled


# -----------------------------
# PCA dimensionality reduction
# -----------------------------
def apply_pca(X, X_test):

    print("Applying PCA...")

    pca = PCA(n_components=40, random_state=42)

    X_pca = pca.fit_transform(X)
    X_test_pca = pca.transform(X_test)

    print("PCA components:", X_pca.shape[1])
    print("Explained variance:", np.sum(pca.explained_variance_ratio_))

    return X_pca, X_test_pca


# -----------------------------
# ElasticNet KFold ensemble
# -----------------------------
def elasticnet_kfold(X, y, X_test):

    print("\nRunning ElasticNet KFold...")

    kf = KFold(n_splits=8, shuffle=True, random_state=42)

    alphas = np.logspace(-3, 0, 20)

    best_alpha = None
    best_score = np.inf

    for alpha in alphas:

        fold_scores = []

        for train_idx, val_idx in kf.split(X):

            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = ElasticNet(
                alpha=alpha,
                l1_ratio=0.5,
                max_iter=20000,
                tol=1e-3,
                selection="random",
                random_state=42
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_val)

            rmse = np.sqrt(mean_squared_error(y_val, preds))
            fold_scores.append(rmse)

        mean_rmse = np.mean(fold_scores)

        print(f"alpha={alpha:.5f} RMSE={mean_rmse:.4f}")

        if mean_rmse < best_score:
            best_score = mean_rmse
            best_alpha = alpha

    print("\nBest alpha:", best_alpha)
    print("Best CV RMSE:", best_score)

    # train ensemble
    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        X_train = X[train_idx]
        y_train = y[train_idx]

        model = ElasticNet(
            alpha=best_alpha,
            l1_ratio=0.5,
            max_iter=20000,
            tol=1e-3,
            selection="random",
            random_state=42
        )

        model.fit(X_train, y_train)

        fold_pred = model.predict(X_test)

        test_preds += fold_pred / kf.n_splits

        print(f"Fold {fold+1} complete")

    return test_preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    name = "exp17_pca_elasticnet_kfold_20260325"

    path = f"../submissions/{name}.csv"

    submission.to_csv(path, index=False, header=False)

    print("\nSaved submission:", path)

    print(pd.read_csv(path, header=None).head())


# -----------------------------
# Pipeline
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = apply_savgol(X, X_test)

    X, X_test = scale_features(X, X_test)

    X, X_test = apply_pca(X, X_test)

    preds = elasticnet_kfold(X, y, X_test)

    save_submission(test, preds)


main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Spectral features: 1555
Applying Savitzky–Golay smoothing...
Applying PCA...
PCA components: 40
Explained variance: 0.9999990143454208

Running ElasticNet KFold...
alpha=0.00100 RMSE=13.8281
alpha=0.00144 RMSE=14.2730
alpha=0.00207 RMSE=14.7236
alpha=0.00298 RMSE=15.1710
alpha=0.00428 RMSE=15.6109
alpha=0.00616 RMSE=16.0406
alpha=0.00886 RMSE=16.4581
alpha=0.01274 RMSE=16.8627
alpha=0.01833 RMSE=17.2569
alpha=0.02637 RMSE=17.6465
alpha=0.03793 RMSE=18.0407
alpha=0.05456 RMSE=18.4503
alpha=0.07848 RMSE=18.8835
alpha=0.11288 RMSE=19.3477
alpha=0.16238 RMSE=19.8451
alpha=0.23357 RMSE=20.3704
alpha=0.33598 RMSE=20.9124
alpha=0.48329 RMSE=21.4590
alpha=0.69519 RMSE=22.0084
alpha=1.00000 RMSE=22.5637

Best alpha: 0.001
Best CV RMSE: 13.828091543378958
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
Fold 6 complete
Fold 7 complete
Fold 8 complete

Saved submission: ../submissions/exp17_pca_elasti